# Name That LLM: TF-IDF baseline with prompt-grouped CV

**Task.** Each passage (40-250 words) was written by one of 18 frontier models. Predict which one.

**Metric.** Balanced accuracy: the mean per-model recall, so every model counts equally no matter how many test passages it has. Chance is 1/18 = 5.6%.

**The catch.** Test passages answer *prompts that never appear in train*. A validation split that lets the same prompt sit in both folds does not match that setting, so its score cannot be trusted. Here it turns out to be badly *pessimistic* (section 2). This notebook:

1. looks at the data that matters for that (prompts, classes, length),
2. measures how far a naive split is from the grouped one,
3. builds a fast character + word TF-IDF linear model validated with `StratifiedGroupKFold` on `prompt_id`,
4. writes `submission.parquet` for the hidden rerun (CPU only, internet off).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, recall_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.svm import LinearSVC

SEED = 42
DATA = Path(os.environ.get("NTL_DATA", "/kaggle/input/competitions/name-that-llm"))
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
LABELS = sorted(train["model"].unique())
print(train.shape, test.shape, f"{len(LABELS)} models, {train['prompt_id'].nunique()} prompts")
train.head()

## 1. What the training set looks like

In [ ]:
counts = train["model"].value_counts()
per_prompt = train.groupby("prompt_id")["model"].agg(["size", "nunique"])
print("passages per model:  min", counts.min(), " max", counts.max())
print("passages per prompt: min", per_prompt["size"].min(), " median", per_prompt["size"].median(), " max", per_prompt["size"].max())
print("models per prompt:   min", per_prompt["nunique"].min(), " median", per_prompt["nunique"].median())

fig, ax = plt.subplots(figsize=(9, 4.5))
counts.sort_values().plot.barh(ax=ax, color="#4c78a8")
ax.set_xlabel("training passages")
ax.set_title("Passages per model")
plt.tight_layout()
plt.show()

Prompts are shared across models, so the same question is answered by many of them. That is what makes topic words useless for telling models apart, and why the folds below are grouped by prompt.

### Is length a shortcut?
The hosts drew passage length at random. A quick check: how well does word count alone classify?

In [ ]:
train["n_words"] = train["text"].str.split().str.len()
order = train.groupby("model")["n_words"].median().sort_values().index
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.boxplot([train.loc[train["model"] == m, "n_words"] for m in order], showfliers=False)
ax.set_xticks(range(1, len(order) + 1), order, rotation=45, ha="right")
ax.set_ylabel("words per passage")
ax.set_title("Passage length by model")
plt.tight_layout()
plt.show()

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(cv.split(train, train["model"], groups=train["prompt_id"]))
oof_len = np.empty(len(train), dtype=object)
for tr, va in folds:
    lr = LogisticRegression(max_iter=2000, class_weight="balanced")
    x = np.log(train[["n_words"]].to_numpy())
    lr.fit(x[tr], train["model"].iloc[tr])
    oof_len[va] = lr.predict(x[va])
print(f"length-only balanced accuracy: {balanced_accuracy_score(train['model'], oof_len):.4f}  (chance {1/18:.4f})")

### Stylistic fingerprints
Punctuation and formatting habits are the kind of signal that survives a change of topic. Rates per 100 words for a few hand-picked markers:

In [ ]:
markers = {  # regex patterns
    "em dash —": "—",
    "en dash –": "–",
    "semicolon ;": ";",
    "colon :": ":",
    "curly apostrophe ’": "’",
    "straight apostrophe '": "'",
    "parenthesis (": r"\(",
    "markdown * #": r"[*#]",
    "LaTeX $ \\": r"[$\\]",
}
rates = pd.DataFrame({
    name: train["text"].str.count(pat) / train["n_words"] * 100 for name, pat in markers.items()
}).groupby(train["model"]).mean()

fig, ax = plt.subplots(figsize=(10, 7))
z = (rates - rates.mean()) / rates.std()
im = ax.imshow(z.to_numpy(), cmap="RdBu_r", vmin=-2.5, vmax=2.5, aspect="auto")
ax.set_xticks(range(len(z.columns)), z.columns, rotation=40, ha="right")
ax.set_yticks(range(len(z.index)), z.index)
for i in range(z.shape[0]):
    for j in range(z.shape[1]):
        ax.text(j, i, f"{rates.iat[i, j]:.2f}", ha="center", va="center", fontsize=7)
ax.set_title("Marker rate per 100 words (colour = z-score across models)")
fig.colorbar(im, ax=ax, shrink=0.6)
plt.tight_layout()
plt.show()

Character n-grams pick these habits up automatically (an em dash, a curly apostrophe, `**` are all just characters), so the model below does not use hand-made features. Case and punctuation are kept: lowercasing or stripping them would throw this signal away.

## 2. Validation: why the split matters
Same model, two splits. `StratifiedKFold` lets a prompt appear in train and validation; `StratifiedGroupKFold` keeps each prompt in one fold, like the real test set.

In [ ]:
def features(fit_texts, *other):
    word = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), lowercase=False, token_pattern=r"(?u)\b\w+\b|[^\w\s]",
                           sublinear_tf=True, min_df=2)
    char = TfidfVectorizer(analyzer="char", ngram_range=(1, 5), lowercase=False, sublinear_tf=True, min_df=2)
    fitted = hstack([word.fit_transform(fit_texts), char.fit_transform(fit_texts)]).tocsr()
    return [fitted] + [hstack([word.transform(t), char.transform(t)]).tocsr() for t in other]


def run_cv(make_model, splits):
    oof = np.empty(len(train), dtype=object)
    for tr, va in splits:
        x_tr, x_va = features(train["text"].iloc[tr], train["text"].iloc[va])
        model = make_model().fit(x_tr, train["model"].iloc[tr])
        oof[va] = model.predict(x_va)
    return oof


svc = lambda: LinearSVC(C=0.5, class_weight="balanced", random_state=SEED)
naive = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(train, train["model"]))
results = {}
for name, splits in [("StratifiedKFold (prompt leak)", naive), ("StratifiedGroupKFold by prompt", folds)]:
    oof = run_cv(svc, splits)
    results[name] = balanced_accuracy_score(train["model"], oof)
    print(f"{name:32s} balanced accuracy = {results[name]:.4f}")

The naive split scores far *lower* (0.146 vs 0.338 on the full training data), the opposite of the usual leak. Each prompt is answered by many models, so in a naive split the training passages on the same topic come from *other* models, and topic n-grams pull the prediction towards the wrong class. With prompts grouped, there is no same-topic neighbour to be misled by, which is also the leaderboard's setting.

The grouped number is the one to trust. Everything below uses the grouped folds.

## 3. Model choice
Two linear models on the same word (1-2) + char (1-5) TF-IDF features, both with `class_weight="balanced"` because the metric weights every model equally.

In [ ]:
candidates = {
    "LinearSVC C=0.5": svc,
    "LogisticRegression C=20": lambda: LogisticRegression(C=20, max_iter=3000, class_weight="balanced"),
}
oofs = {}
for name, make in candidates.items():
    oofs[name] = run_cv(make, folds)
    print(f"{name:26s} grouped-CV balanced accuracy = {balanced_accuracy_score(train['model'], oofs[name]):.4f}")
best = max(oofs, key=lambda k: balanced_accuracy_score(train["model"], oofs[k]))
print("best:", best)

## 4. Where it goes wrong

In [ ]:
oof = oofs[best]
recall = pd.Series(recall_score(train["model"], oof, labels=LABELS, average=None), index=LABELS).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={"width_ratios": [1, 1.6]})
recall.plot.barh(ax=axes[0], color="#4c78a8")
axes[0].axvline(1 / 18, color="grey", ls="--", lw=1)
axes[0].set_title("Out-of-fold recall per model")

cm = confusion_matrix(train["model"], oof, labels=LABELS, normalize="true")
axes[1].imshow(cm, cmap="Blues", vmin=0, vmax=1)
axes[1].set_xticks(range(len(LABELS)), LABELS, rotation=90)
axes[1].set_yticks(range(len(LABELS)), LABELS)
axes[1].set_xlabel("predicted")
axes[1].set_ylabel("true")
axes[1].set_title("Row-normalised confusion matrix")
plt.tight_layout()
plt.show()

vendor = lambda s: pd.Series(s).str.extract(r"^(gpt|gemini|claude|qwen|glm|kimi)")[0]
print(f"vendor-level accuracy: {(vendor(train['model']).to_numpy() == vendor(oof).to_numpy()).mean():.4f}")

## 5. Fit on all of train and write the submission

In [ ]:
x_train, x_test = features(train["text"], test["text"])
final = candidates[best]().fit(x_train, train["model"])
submission = pd.DataFrame({"id": test["id"], "model": final.predict(x_test)})

assert submission["id"].is_unique and len(submission) == len(test)
assert submission.notna().all().all()
assert set(submission["model"]) <= set(LABELS) and len(LABELS) == 18
submission.to_parquet("submission.parquet", index=False)
print(submission.shape)
submission.head()

## Results (measured on the full training data, grouped 5-fold CV)

| Model | Balanced accuracy |
|---|---|
| Chance (1/18) | 0.056 |
| Passage length only | 0.058 |
| Word + char TF-IDF, LinearSVC (C=0.5) | 0.338 |
| Word + char TF-IDF, LogisticRegression (C=20) | **0.349** |

Same TF-IDF + LinearSVC with a naive `StratifiedKFold`: 0.146. Vendor-level accuracy of the best model: 0.704, so most errors are between models from the same lab.


## Where to go next
- **Probability-level ensembling** of the char and word views (or a calibrated SVC) instead of one stacked matrix.
- **Pretrained encoders** (DeBERTa-v3 and friends) fine-tuned with the same grouped folds; they are allowed offline as Kaggle datasets.
- **Hierarchical decoding**: the confusion matrix shows which confusions dominate; a vendor-then-model scheme may help there.
- **Prior shift**: the test model mix differs from train and between public/private LB, so avoid tuning to class frequencies.